# שבוע 06: ניתוח רכיבים ראשיים (PCA)

לאחר GPA, המרחב בעל ממדים רבים (2×16=32 משתנים) — PCA מצמצם אותו  
לרכיבים עיקריים שמסבירים את רוב השונות בצורה.

**מטרות השיעור:**
- הרצת PCA על נתוני GPA
- פרשנות תרשים ה-Scree
- גרף פיזור PCA עם צביעה לפי קיסר
- פרשנות צורות קיצוניות לאורך הצירים

In [ ]:
!pip install morphops python-bidi -q

import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
import morphops as mops
rtl = get_display
print('הכל מוכן!')

In [ ]:
def parse_tps(filepath_or_text):
    """Works with both local path string and text content."""
    if '\n' in filepath_or_text:
        lines = filepath_or_text.strip().split('\n')
    else:
        with open(filepath_or_text, encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
    specimens, ids = [], []
    i = 0
    while i < len(lines):
        line = lines[i].strip() if hasattr(lines[i], 'strip') else lines[i]
        if line.startswith('LM='):
            n_lm = int(line.split('=')[1])
            coords = []
            for j in range(n_lm):
                i += 1
                parts = lines[i].strip().replace(',', '.').split()
                coords.append([float(parts[0]), float(parts[1])])
            specimens.append(np.array(coords))
        elif line.startswith('ID='):
            ids.append(line.split('=')[1].strip())
        i += 1
    return np.array(specimens), ids

print('parse_tps מוכן')

In [ ]:
import urllib.request

BASE = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/coins/'

def load_tps_url(url):
    with urllib.request.urlopen(url, timeout=15) as r:
        return r.read().decode('utf-8', errors='replace')

try:
    had_text = load_tps_url(BASE + 'hadrian.tps')
    ant_text = load_tps_url(BASE + 'antoninus.tps')
    had_lm, had_ids = parse_tps(had_text)
    ant_lm, ant_ids = parse_tps(ant_text)
    print(f'הדריאנוס: {len(had_lm)} מטבעות, {had_lm.shape[1]} נקודות ציון')
    print(f'אנטונינוס פיוס: {len(ant_lm)} מטבעות, {ant_lm.shape[1]} נקודות ציון')
    DATA_OK = True
except Exception as e:
    print(f'שגיאה: {e} — משתמשים בנתונים סינתטיים')
    np.random.seed(42)
    n_lm = 16
    angles = np.linspace(0, 2*np.pi, n_lm, endpoint=False)
    base = np.column_stack([np.cos(angles)*100, np.sin(angles)*80])
    had_lm = np.array([base + np.random.randn(n_lm, 2)*5 for _ in range(22)])
    ant_lm = np.array([base * 0.9 + np.random.randn(n_lm, 2)*5 + [10, 5] for _ in range(15)])
    had_ids = [f'Hadrian_{i+1}' for i in range(22)]
    ant_ids = [f'Antoninus_{i+1}' for i in range(15)]
    DATA_OK = False

all_lm = np.concatenate([had_lm, ant_lm], axis=0)
labels = np.array(['Hadrian']*len(had_lm) + ['Antoninus']*len(ant_lm))
print(f'\nסה"כ: {len(all_lm)} מטבעות, תוויות: {np.unique(labels, return_counts=True)}')

In [ ]:
result = mops.gpa(all_lm)
aligned = result['aligned']
mean_shape = result['mean']

# שטח לווקטורי צורה
n = len(aligned)
shape_matrix = aligned.reshape(n, -1)  # (n_specimens, n_lm*2)
print(f'מטריצת צורה: {shape_matrix.shape}')

## PCA על מרחב הצורה

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
scores = pca.fit_transform(shape_matrix)
variance_explained = pca.explained_variance_ratio_ * 100

print('שונות מוסברת — 5 רכיבים ראשונים:')
for i, v in enumerate(variance_explained[:5]):
    print(f'  PC{i+1}: {v:.1f}%')
print(f'  PC1+PC2 יחד: {variance_explained[:2].sum():.1f}%')

## תרשים Scree

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scree
n_show = min(15, len(variance_explained))
ax = axes[0]
ax.bar(range(1, n_show+1), variance_explained[:n_show],
       color='steelblue', alpha=0.8)
ax.plot(range(1, n_show+1), variance_explained[:n_show],
        'ro-', markersize=5)
ax.set_xlabel('רכיב ראשי (PC)', fontsize=11)
ax.set_ylabel('שונות מוסברת (%)', fontsize=11)
ax.set_title(rtl('תרשים Scree'), fontsize=13)
ax.set_xticks(range(1, n_show+1))

# שונות מצטברת
ax = axes[1]
cumvar = np.cumsum(variance_explained[:n_show])
ax.plot(range(1, n_show+1), cumvar, 'gs-', markersize=6)
ax.axhline(80, color='red', linestyle='--', alpha=0.7, label='80%')
ax.axhline(95, color='orange', linestyle='--', alpha=0.7, label='95%')
ax.set_xlabel('מספר רכיבים', fontsize=11)
ax.set_ylabel('שונות מצטברת (%)', fontsize=11)
ax.set_title(rtl('שונות מצטברת'), fontsize=13)
ax.legend()
ax.set_xticks(range(1, n_show+1))

plt.tight_layout()
plt.show()

## גרף פיזור PCA: PC1 × PC2

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

mask_h = labels == 'Hadrian'
mask_a = labels == 'Antoninus'

ax.scatter(scores[mask_h, 0], scores[mask_h, 1],
           c='steelblue', s=80, alpha=0.8, label=f'Hadrian (n={mask_h.sum()})', zorder=3)
ax.scatter(scores[mask_a, 0], scores[mask_a, 1],
           c='coral', s=80, marker='s', alpha=0.8,
           label=f'Antoninus (n={mask_a.sum()})', zorder=3)

ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.axvline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlabel(f'PC1 ({variance_explained[0]:.1f}%)', fontsize=12)
ax.set_ylabel(f'PC2 ({variance_explained[1]:.1f}%)', fontsize=12)
ax.set_title(rtl('מרחב הצורה: מטבעות רומיים (PCA)'), fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## צורות קיצוניות לאורך PC1

נחשב את הצורה ב-−2SD, בממוצע ו-+2SD של PC1 כדי להבין מה הציר הזה מייצג.

In [ ]:
sd1 = np.sqrt(pca.explained_variance_[0])
n_lm = mean_shape.shape[0]

# PC1 eigenvector
pc1 = pca.components_[0].reshape(n_lm, 2)

shapes = {
    '-2SD': mean_shape + pc1 * (-2 * sd1),
    'ממוצע': mean_shape.copy(),
    '+2SD': mean_shape + pc1 * (2 * sd1)
}

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
colors_map = {'-2SD': 'steelblue', 'ממוצע': 'black', '+2SD': 'coral'}

for ax, (label, shape) in zip(axes, shapes.items()):
    ax.scatter(shape[:, 0], shape[:, 1],
               color=colors_map[label], s=60, zorder=3)
    ax.plot(np.append(shape[:, 0], shape[0, 0]),
            np.append(shape[:, 1], shape[0, 1]),
            color=colors_map[label], lw=1)
    # קווי רשת עדינים מהממוצע
    for j in range(n_lm):
        ax.plot([mean_shape[j, 0], shape[j, 0]],
                [mean_shape[j, 1], shape[j, 1]],
                'gray', lw=0.5, alpha=0.5)
    ax.set_title(rtl(label), fontsize=12)
    ax.set_aspect('equal')
    ax.axis('off')

plt.suptitle(rtl('שינוי צורה לאורך PC1 (−2SD, ממוצע, +2SD)'), fontsize=13)
plt.tight_layout()
plt.show()

## סיכום

- **PCA** מצמצם 32 משתני צורה לרכיבים עצמאיים
- **Scree plot**: כמה רכיבים מספיקים? בדרך כלל שמים רכיבים עד 80-90% שונות
- **גרף פיזור**: מאפשר לראות הפרדה חזותית בין קבוצות
- **צורות קיצוניות**: מסבירות *מה* כל ציר מייצג

**שאלות לחשיבה:**
1. כמה רכיבים נחוצים כדי להסביר 80% מהשונות?
2. האם המטבעות מופרדים בצורה ברורה? מה המשמעות?
3. מה הצורה ב-+2SD של PC2?